In [92]:
import pandas as pd

In [93]:
df = pd.read_csv('../data/challenge_17.csv')
df.head()

,RecordID,Open Date,Close Date
0,1,"April 03, 2013","May 06, 2013"
1,2,"April 14, 2013",NaN
2,3,"May 03, 2013","July 18, 2013"
3,4,"May 24, 2013","June 12, 2013"
4,5,"June 13, 2013","July 10, 2013"


In [94]:
# parsing dates
df['Open Date'] = pd.to_datetime(df['Open Date'], format='%B %d, %Y')
df['Close Date'] = pd.to_datetime(df['Close Date'], format='%B %d, %Y')
df.head()

,RecordID,Open Date,Close Date
0,1,2013-04-03,2013-05-06
1,2,2013-04-14,NaT
2,3,2013-05-03,2013-07-18
3,4,2013-05-24,2013-06-12
4,5,2013-06-13,2013-07-10


In [95]:
# create list of months as datetimes
months = ['2013-05-01', '2013-06-01', '2013-07-01', '2013-08-01']

for i in range(len(months)):
    months[i] = pd.to_datetime(months[i])

months

[Timestamp('2013-05-01 00:00:00'),
 Timestamp('2013-06-01 00:00:00'),
 Timestamp('2013-07-01 00:00:00'),
 Timestamp('2013-08-01 00:00:00')]

In [96]:
# a df will be added to this list each time the loop runs. The results will then be unioned
results = []

for month in months:
    # create working copy of df and add the start of month as a column
    w_df = df.copy()
    w_df['start of month'] = month

    # filter the df to the accounts open 24 months before the start of the given month
    w_df = w_df[(
            w_df['Open Date'] > w_df['start of month'] - pd.offsets.DateOffset(years=2))
            & (w_df['Open Date'] < w_df['start of month'])
            & ((w_df['Close Date'] > w_df['start of month'])
            | w_df['Close Date'].isna())
        ]
    
    # create df with only closed accounts that month
    c_df = w_df[w_df['Close Date'] < w_df['start of month'] + pd.offsets.DateOffset(months=1)]

    # aggregate and join the dfs
    c_df = c_df.groupby('start of month')['Close Date'].count().reset_index().rename(columns={'Close Date': 'close'})
    w_df = w_df.groupby('start of month')['Open Date'].count().reset_index().rename(columns={'Open Date': 'open'})
    output_df = pd.merge(c_df, w_df, how='inner', on='start of month')

    # union result to final_df
    results.append(output_df)

# union all results into a single df
final_df = pd.concat(results, axis=0)
final_df

,start of month,close,open
0,2013-05-01,1,2
0,2013-06-01,1,3
0,2013-07-01,2,4
0,2013-08-01,1,5
